<div style="text-align:center; background-color:#fff6e4; padding:20px; border:5px solid #f5ecda; border-radius:8px;">
    <div style="font-size:36px; font-weight:bold; color:#4A4A4A;">
        Buscando Nombre
    </div>
    <div style="font-size:24px; font-weight:bold; color:#4A4A4A;">
        Part 1: EDA &amp; Preprocessing
    </div>
    <div style="font-size:14px; font-weight:normal; color:#666; margin-top:16px;">
        Author: Jerónimo Hoyos Botero <br> 
        Created: May 2026<br>
        Last updated: May 2026
    </div>
</div>

<div style="background-color:#2c699d; color:white; padding:15px; border-radius:6px;">
    <h1 style="margin:0px">Setup</h1>
</div>

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Notebook Settings</strong>
</div>

In [1]:
# Automatically reload local modules before each cell run
%load_ext autoreload
%autoreload 2

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Imports</strong>
</div>

In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import math

<div style="background-color:#2c699d; color:white; padding:15px; border-radius:6px;">
    <h1 style="margin:0px">Data Loading</h1>
</div>

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Load datasets from GitHub</strong>
</div>

In [3]:
base = "https://raw.githubusercontent.com/Semillero-Inteligencia-Artificial-EAFIT/p-REDbloodDON/main/data/"

banco = pd.read_csv(
    base + "Banco_de_sangre,_Hospital_General_de_Medell%C3%ADn_20260508.csv",
    encoding="latin-1",
)
defunciones = pd.read_csv(
    base
    + "Defunciones_ocurridas_en__en_el_Hospital_General_de_Medell%C3%ADn_20260508.csv",
    encoding="latin-1",
)
poblacion1 = pd.read_csv(
    base
    + "Poblaci%C3%B3n_atendida_en_el_Hospital_General_de_Medell%C3%ADn_20260508.csv",
    encoding="latin-1",
)
poblacion2 = pd.read_csv(
    base + "poblacion_atendida_hospital_general.csv", encoding="latin-1"
)

print("Banco:      ", banco.shape)
print("Defunciones:", defunciones.shape)
print("Población 1:", poblacion1.shape)
print("Población 2:", poblacion2.shape)

HTTPError: HTTP Error 404: Not Found

<div style="background-color:#2c699d; color:white; padding:15px; border-radius:6px;">
    <h1 style="margin:0px">Data Diagnostic</h1>
</div>

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Columns & sample rows</strong>
</div>

In [ ]:
for nombre, df in [
    ("Banco", banco),
    ("Defunciones", defunciones),
    ("Población 1", poblacion1),
    ("Población 2", poblacion2),
]:
    print(f"{'─'*40}")
    print(f"  {nombre}")
    print(f"{'─'*40}")
    print(df.columns.tolist())
    print(df.head(2))

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Data types & null counts</strong>
</div>

In [ ]:
for nombre, df in [
    ("Banco", banco),
    ("Defunciones", defunciones),
    ("Población 1", poblacion1),
    ("Población 2", poblacion2),
]:
    print(f"\n{'─'*60}")
    print(f"INFO: {nombre}")
    print(f"{'─'*60}")
    df.info()

<div style="background-color:#2c699d; color:white; padding:15px; border-radius:6px;">
    <h1 style="margin:0px">Preprocessing</h1>
</div>

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Column cleaning & deduplication</strong>
</div>

In [ ]:
def limpiar(df):
    df.columns = (
        df.columns.str.strip()
        .str.encode("latin-1", errors="ignore")
        .str.decode("utf-8", errors="ignore")
        .str.lower()
        .str.replace(" ", "_")
    )
    # Rename columns with 'ao' or 'ano' → 'año'
    df.columns = [col.replace("ao", "año").replace("ano", "año") for col in df.columns]
    df = df.drop_duplicates()
    return df


banco       = limpiar(banco)
defunciones = limpiar(defunciones)
poblacion1  = limpiar(poblacion1)
poblacion2  = limpiar(poblacion2)

# Fill missing RH with mode
banco["rh"] = banco["rh"].fillna(banco["rh"].mode()[0])

print(banco.columns.tolist())
print(defunciones.columns.tolist())
print(poblacion1.columns.tolist())
print(poblacion2.columns.tolist())

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Standardize column names & parse dates</strong>
</div>

In [ ]:
# Align column names between the two population tables
poblacion2 = poblacion2.rename(
    columns={
        "periodo_reporte":    "periodo_de_reporte",
        "fecha_atencion":     "fecha_atencion",
        "rango_edad":         "rango_de_edad",
        "codigo_aseguradora": "cod._aseguradora",
    }
)

# Parse dates
banco["fecha_extraccion"] = pd.to_datetime(
    banco["fecha_extraccion"], dayfirst=True, errors="coerce"
)

defunciones["fecha_defuncion"] = pd.to_datetime(
    defunciones["fecha_defuncion"], dayfirst=True, errors="coerce"
)

poblacion1["fecha_atencion"] = pd.to_datetime(
    poblacion1["fecha_atencion"], dayfirst=True, errors="coerce"
)

poblacion2["fecha_atencion"] = pd.to_datetime(
    poblacion2["fecha_atencion"], format="%Y-%m-%dT%H:%M:%S.%f%z", errors="coerce"
).dt.tz_localize(None)

# Merge population tables
poblacion = pd.concat([poblacion1, poblacion2], ignore_index=True)
poblacion = poblacion.drop_duplicates()
print("Población unificada:", poblacion.shape)

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Save cleaned datasets</strong>
</div>

In [ ]:
banco.to_csv("banco_limpio.csv", index=False)
defunciones.to_csv("defunciones_limpias.csv", index=False)
poblacion.to_csv("poblacion_unido.csv", index=False)

<div style="background-color:#2c699d; color:white; padding:15px; border-radius:6px;">
    <h1 style="margin:0px">Monthly Time Series</h1>
</div>

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Build monthly series</strong>
</div>

In [ ]:
# Donations per month
serie_banco = (
    banco.groupby(pd.Grouper(key="fecha_extraccion", freq="ME"))
    .size()
    .reset_index(name="donaciones")
)

# Deaths per month
serie_def = (
    defunciones.groupby(pd.Grouper(key="fecha_defuncion", freq="ME"))
    .size()
    .reset_index(name="defunciones")
)

# Attended population per month
serie_pob = (
    poblacion.groupby(pd.Grouper(key="fecha_atencion", freq="ME"))
    .size()
    .reset_index(name="atenciones")
)

# Rename date columns and merge
serie_def   = serie_def.rename(columns={"fecha_defuncion": "fecha"})
serie_pob   = serie_pob.rename(columns={"fecha_atencion":  "fecha"})
serie_banco = serie_banco.rename(columns={"fecha_extraccion": "fecha"})

serie = serie_banco.merge(serie_def, on="fecha", how="left").merge(
    serie_pob, on="fecha", how="left"
)

print(serie.tail(10))

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Plot monthly series</strong>
</div>

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

sns.lineplot(
    data=serie, x="fecha", y="donaciones",
    ax=axes[0], color="steelblue", linewidth=2, marker="o", markersize=4,
)
axes[0].set_title("Donaciones mensuales", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Cantidad", fontsize=11)
axes[0].grid(True, alpha=0.3)

sns.lineplot(
    data=serie, x="fecha", y="defunciones",
    ax=axes[1], color="tomato", linewidth=2, marker="o", markersize=4,
)
axes[1].set_title("Defunciones mensuales", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Cantidad", fontsize=11)
axes[1].grid(True, alpha=0.3)

sns.lineplot(
    data=serie, x="fecha", y="atenciones",
    ax=axes[2], color="seagreen", linewidth=2, marker="o", markersize=4,
)
axes[2].set_title("Atenciones mensuales", fontsize=14, fontweight="bold")
axes[2].set_ylabel("Cantidad", fontsize=11)
axes[2].set_xlabel("Fecha", fontsize=11)
axes[2].grid(True, alpha=0.3)

axes[2].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.setp(axes[2].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Detect incomplete months</strong>
</div>

In [ ]:
# Flag months with abnormally low donations (below p10)
umbral = serie["donaciones"].quantile(0.10)
serie["dato_incompleto"] = (serie["donaciones"] < umbral).astype(int)
print(serie[serie["dato_incompleto"] == 1][["fecha", "donaciones"]])

<div style="background-color:#e8f4fd; padding:15px; border:3px solid #d0e7fa; border-radius:6px;">
    <strong style="color:#000000;">Date ranges per dataset</strong>
</div>

In [ ]:
print("Banco:      ", banco["fecha_extraccion"].min(), "→", banco["fecha_extraccion"].max())
print("Defunciones:", defunciones["fecha_defuncion"].min(), "→", defunciones["fecha_defuncion"].max())
print("Población:  ", poblacion["fecha_atencion"].min(), "→", poblacion["fecha_atencion"].max())